# LeetCode 1901 — Find a Peak Element II bằng Python

Link đề: https://leetcode.com/problems/find-a-peak-element-ii/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def findPeakGrid(self, mat: List[List[int]]) -> List[int]:
        ...
```

Lộ trình (bài có **nhiều đáp án đúng** — ta kiểm tra bằng hàm `is_peak`):
1. **Quét mọi ô** và kiểm tra 4 láng giềng (`O(m n)`).
2. **Phần tử lớn nhất toàn ma trận** luôn là đỉnh (`O(m n)`).
3. **Leo đồi (hill climbing)**: từ một ô, luôn bước sang láng giềng lớn hơn — dừng ở đỉnh (`O(m n)` worst).
4. **Tìm kiếm nhị phân trên hàng**: ở hàng giữa lấy cột có giá trị lớn nhất, so với ô trên/dưới để chọn nửa — `O(n log m)` (yêu cầu của đề).
5. **Tìm kiếm nhị phân trên cột** (đối xứng) — `O(m log n)`; chọn chiều nào nhỏ hơn để có `O(min(m, n) · log max(m, n))`.

## 1. Mục tiêu học

1. Hiểu khái niệm **đỉnh** trong lưới 2D (lớn hơn nghiêm ngặt 4 láng giềng, biên coi như `−1`) và vì sao **luôn tồn tại** đỉnh.
2. Mở rộng **tìm kiếm nhị phân tìm đỉnh 1D** (LeetCode 162) lên 2D bằng cách nén mỗi hàng về **giá trị lớn nhất** của nó.
3. Chứng minh bất biến: "nửa được chọn luôn chứa một đỉnh **của toàn ma trận**".
4. Viết hàm **kiểm tra hợp lệ** cho bài nhiều đáp án và dùng nó để test mọi cách.
5. So sánh thực nghiệm `O(m n)` với `O(n log m)` trên lưới `500 × 500`.

## 2. Đề bài & ví dụ

Một **phần tử đỉnh** trong lưới 2D là phần tử **lớn hơn nghiêm ngặt** mọi láng giềng **trái, phải, trên, dưới**.

Cho ma trận `mat` kích thước `m × n` (chỉ số từ 0) trong đó **không có hai ô kề nhau bằng nhau**, tìm **một** phần tử đỉnh `mat[i][j]` bất kỳ và trả về `[i, j]`.

Giả sử cả ma trận được bao quanh bởi một viền giá trị `−1`.

Yêu cầu: thuật toán chạy trong `O(m log n)` hoặc `O(n log m)`.

**Ví dụ 1:** `mat = [[1,4],[3,2]]` → `[0,1]` (cả `3` và `4` đều là đỉnh; `[1,0]` cũng được chấp nhận).

**Ví dụ 2:** `mat = [[10,20,15],[21,30,14],[7,16,32]]` → `[1,1]` (`30` và `32` đều là đỉnh; `[2,2]` cũng đúng).

**Ràng buộc:** `1 ≤ m, n ≤ 500`, `1 ≤ mat[i][j] ≤ 10⁵`, không có hai ô kề nhau bằng nhau.

In [1]:
import random
import time
from typing import List

examples = [
    ([[1, 4], [3, 2]], [[0, 1], [1, 0]]),
    ([[10, 20, 15], [21, 30, 14], [7, 16, 32]], [[1, 1], [2, 2]]),
]

def is_peak(mat, i, j):
    m, n = len(mat), len(mat[0])
    v = mat[i][j]
    for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        a, b = i + di, j + dj
        nb = mat[a][b] if 0 <= a < m and 0 <= b < n else -1
        if nb >= v:
            return False
    return True

for mat, ok in examples:
    print(f"mat={mat} -> đáp án hợp lệ: {ok}; kiểm tra:", [is_peak(mat, i, j) for i, j in ok])

mat=[[1, 4], [3, 2]] -> đáp án hợp lệ: [[0, 1], [1, 0]]; kiểm tra: [True, True]
mat=[[10, 20, 15], [21, 30, 14], [7, 16, 32]] -> đáp án hợp lệ: [[1, 1], [2, 2]]; kiểm tra: [True, True]


## 3. Kiến thức nền

### 3.1 Láng giềng 4 hướng và viền `−1`

Hàm `get(i, j)` trả `−1` khi ra ngoài lưới giúp xử lý biên gọn gàng.

In [2]:
def get(mat, i, j):
    return mat[i][j] if 0 <= i < len(mat) and 0 <= j < len(mat[0]) else -1

mat = examples[1][0]
print("láng giềng của (1,1):", [get(mat, 0, 1), get(mat, 2, 1), get(mat, 1, 0), get(mat, 1, 2)])

láng giềng của (1,1): [20, 16, 21, 14]


### 3.2 Đỉnh 1D bằng tìm kiếm nhị phân (LeetCode 162)

Với mảng không có hai phần tử kề bằng nhau và biên `−∞`: nếu `a[mid] < a[mid + 1]` thì bên phải **chắc chắn** có đỉnh (đi lên mãi phải dừng ở đâu đó); ngược lại bên trái (gồm `mid`) có đỉnh.

In [3]:
def peak_1d(a):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < a[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

a = [1, 3, 2, 5, 7, 6, 4]
print(peak_1d(a), a[peak_1d(a)])

4 7


### 3.3 Chỉ số của giá trị lớn nhất trong một hàng

`max(range(n), key=row.__getitem__)` hoặc `row.index(max(row))`.

In [4]:
row = [21, 30, 14]
print(row.index(max(row)), max(range(len(row)), key=row.__getitem__))

1 1


### 3.4 Vì sao luôn có đỉnh?

Ô có giá trị **lớn nhất toàn ma trận** lớn hơn (nghiêm ngặt, vì không có ô kề bằng nhau... thật ra chỉ cần lớn hơn hoặc bằng, nhưng do không kề bằng nhau nên nghiêm ngặt) mọi láng giềng và lớn hơn `−1` ⇒ là đỉnh.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2:

```
10 20 15
21 30 14
 7 16 32
```

- Hàng giữa (hàng 1): max là `30` ở cột 1. So với trên `20` và dưới `16`: `30` lớn hơn cả hai ⇒ `30` lớn hơn **mọi** ô trong hàng của nó (vì là max hàng) và lớn hơn trên/dưới ⇒ **đỉnh** `[1, 1]`.

Nếu max của hàng giữa **nhỏ hơn** ô bên dưới, ta "leo" xuống: nửa dưới chắc chắn chứa đỉnh (lập luận ở mục 6).

**Trực giác "leo đồi":** từ một ô bất kỳ, luôn đi sang láng giềng lớn hơn; giá trị tăng nghiêm ngặt nên không lặp, lưới hữu hạn nên phải dừng — dừng ở một đỉnh.

In [5]:
mat = examples[1][0]
mid = 1
j = mat[mid].index(max(mat[mid]))
print(f"hàng {mid}: max {mat[mid][j]} tại cột {j}; trên = {get(mat, mid - 1, j)}, dưới = {get(mat, mid + 1, j)}")

hàng 1: max 30 tại cột 1; trên = 20, dưới = 16


## 5. Cách 1 — Brute Force / baseline: quét mọi ô

In [6]:
def find_peak_scan(mat):
    m, n = len(mat), len(mat[0])
    for i in range(m):
        for j in range(n):
            if is_peak(mat, i, j):
                return [i, j]
    return [-1, -1]

for mat, ok in examples:
    r = find_peak_scan(mat)
    print(r, "hợp lệ?", is_peak(mat, *r))

[0, 1] hợp lệ? True
[1, 1] hợp lệ? True


**Độ phức tạp:** Time `O(m n)` (mỗi ô 4 phép so sánh), Space `O(1)`. Với `500 × 500 = 2.5·10⁵` ô thì chạy nhanh, nhưng **không đạt** yêu cầu `O(n log m)` của đề.

**Nút thắt:** ta xem mọi ô, trong khi chỉ cần "đi theo hướng tăng" một cách thông minh.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (nén hàng):** với hàng `r`, gọi `c = argmax(mat[r])`. So sánh `mat[r][c]` với `mat[r+1][c]`:
- Nếu `mat[r][c] < mat[r+1][c]`: nửa **dưới** (hàng `r+1..m−1`) chứa một đỉnh của toàn ma trận.
- Ngược lại (`mat[r][c] > mat[r+1][c]`): nửa **trên** (hàng `0..r`) chứa một đỉnh.

**Vì sao?** (trường hợp đi xuống) Xét ô lớn nhất `M` trong vùng hàng `r+1..m−1`. `M ≥ mat[r+1][c] > mat[r][c] ≥` mọi ô hàng `r` ⇒ `M` lớn hơn cả ô phía trên nó (nếu `M` ở hàng `r+1`, ô trên thuộc hàng `r`). Trong vùng, `M` lớn hơn mọi láng giềng (là max vùng). Ngoài vùng chỉ có viền `−1` hoặc hàng `r`. ⇒ `M` là đỉnh toàn ma trận.

**Thuật toán:** tìm nhị phân trên hàng như đỉnh 1D, với "giá trị hàng" = max của hàng.

**Invariant:** đoạn hàng `[lo, hi]` luôn chứa một đỉnh của toàn ma trận (mà max của đoạn là đỉnh). Khi `lo == hi`, max của hàng `lo` là đỉnh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Phần tử lớn nhất toàn ma trận

In [7]:
def find_peak_global_max(mat):
    best = (-1, 0, 0)
    for i, row in enumerate(mat):
        for j, v in enumerate(row):
            if v > best[0]:
                best = (v, i, j)
    return [best[1], best[2]]

for mat, ok in examples:
    r = find_peak_global_max(mat)
    print(r, "hợp lệ?", is_peak(mat, *r))

[0, 1] hợp lệ? True
[2, 2] hợp lệ? True


**Độ phức tạp:** Time `O(m n)`, Space `O(1)`. Đơn giản nhất và luôn đúng.

### 7.2 Cách 3 — Leo đồi (hill climbing, đi sang láng giềng lớn nhất)

In [8]:
def find_peak_climb(mat, start=(0, 0)):
    m, n = len(mat), len(mat[0])
    i, j = start
    steps = 0
    while True:
        best = (mat[i][j], i, j)
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and mat[a][b] > best[0]:
                best = (mat[a][b], a, b)
        if (best[1], best[2]) == (i, j):
            return [i, j], steps
        i, j = best[1], best[2]
        steps += 1

for mat, ok in examples:
    r, st = find_peak_climb(mat)
    print(r, "hợp lệ?", is_peak(mat, *r), "sau", st, "bước")

[0, 1] hợp lệ? True sau 1 bước
[1, 1] hợp lệ? True sau 2 bước


**Độ phức tạp:** mỗi bước tăng giá trị nghiêm ngặt ⇒ tối đa `m n` bước. Worst `O(m n)` (ví dụ đường "rắn" tăng dần qua cả lưới), trung bình nhanh hơn nhiều.

### 7.3 Cách 4 — Nhị phân trên hàng `O(n log m)`

Từng đoạn:
1. `lo, hi = 0, m − 1`.
2. `mid = (lo + hi) // 2`, `c = argmax(mat[mid])`.
3. Nếu `mid + 1 < m` và `mat[mid][c] < mat[mid + 1][c]` ⇒ `lo = mid + 1`; ngược lại `hi = mid`.
4. Kết thúc: `[lo, argmax(mat[lo])]`.

In [9]:
def find_peak_rows(mat):
    lo, hi = 0, len(mat) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        row = mat[mid]
        c = row.index(max(row))
        if mat[mid][c] < mat[mid + 1][c]:
            lo = mid + 1
        else:
            hi = mid
    row = mat[lo]
    return [lo, row.index(max(row))]

for mat, ok in examples:
    r = find_peak_rows(mat)
    print(r, "hợp lệ?", is_peak(mat, *r))

[0, 1] hợp lệ? True
[1, 1] hợp lệ? True


Vì `mid < hi ≤ m − 1` nên `mid + 1` luôn hợp lệ trong vòng lặp. Khi kết thúc, hàng `lo` được chọn bằng các so sánh "đi về phía lớn hơn" — max hàng `lo` lớn hơn ô trên/dưới cùng cột (chứng minh ở mục 10).

Code nộp LeetCode của Cách 4: xem mục 9.

**Độ phức tạp:** `log m` vòng × `O(n)` tìm max hàng ⇒ `O(n log m)`, Space `O(1)`.

### 7.4 Cách 5 — Nhị phân trên chiều nhỏ hơn

Nếu `n > m`, chuyển vị ma trận (hoặc nhị phân trên **cột**: ở cột giữa lấy hàng max, so với trái/phải) để được `O(m log n)`. Tổng quát: `O(min(m, n) · log max(m, n))`.

In [10]:
def find_peak_cols(mat):
    m, n = len(mat), len(mat[0])
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        r = max(range(m), key=lambda i: mat[i][mid])
        if mat[r][mid] < mat[r][mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    r = max(range(m), key=lambda i: mat[i][lo])
    return [r, lo]

def find_peak_best_dim(mat):
    return find_peak_rows(mat) if len(mat[0]) <= len(mat) else find_peak_cols(mat)

for mat, ok in examples:
    for fn in (find_peak_cols, find_peak_best_dim):
        r = fn(mat)
        print(fn.__name__, r, "hợp lệ?", is_peak(mat, *r))

find_peak_cols [1, 0] hợp lệ? True
find_peak_best_dim [0, 1] hợp lệ? True
find_peak_cols [1, 1] hợp lệ? True
find_peak_best_dim [1, 1] hợp lệ? True


**Độ phức tạp:** `find_peak_cols`: `O(m log n)`. `find_peak_best_dim`: `O(min(m,n) · log max(m,n))`... chính xác: nhị phân trên chiều **dài** và tìm max trên chiều **ngắn**. (Với `n ≤ m`, nhị phân trên hàng: `O(n log m)`.)

## 8. TRACE từng bước

Trace Cách 4 trên một ma trận `6 × 4` — in hàng giữa, max hàng, ô dưới và quyết định:

In [11]:
def trace_rows(mat):
    lo, hi = 0, len(mat) - 1
    while lo < hi:
        mid = (lo + hi) // 2; row = mat[mid]; c = row.index(max(row))
        down = mat[mid + 1][c]
        go = "xuống (lo = mid + 1)" if row[c] < down else "lên/giữ (hi = mid)"
        print(f"[lo={lo}, hi={hi}] mid={mid}: max hàng = {row[c]} tại cột {c}, ô dưới = {down} -> {go}")
        if row[c] < down: lo = mid + 1
        else: hi = mid
    c = mat[lo].index(max(mat[lo]))
    print(f"kết thúc ở hàng {lo}, cột {c}: giá trị {mat[lo][c]}, đỉnh? {is_peak(mat, lo, c)}")
    return [lo, c]

M = [[1, 2, 3, 4],
     [8, 7, 6, 5],
     [9, 10, 11, 12],
     [16, 15, 14, 13],
     [17, 18, 19, 20],
     [24, 23, 22, 21]]
trace_rows(M)

def snake_walls(m, n):
    """Ma trận có 'đường rắn' tăng dần ở các hàng chẵn, các hàng lẻ là tường giá trị nhỏ (trừ một ô nối).
    Mọi giá trị phân biệt. Leo đồi từ (0,0) buộc phải đi theo cả đường rắn."""
    walls = iter(range(1, m * n + 1))
    g = [[0] * n for _ in range(m)]
    v = m * n + 1
    for i in range(m):
        if i % 2 == 0:
            cols = range(n) if (i // 2) % 2 == 0 else range(n - 1, -1, -1)
            for j in cols:
                g[i][j] = v; v += 1
        else:
            conn = n - 1 if (i // 2) % 2 == 0 else 0
            for j in range(n):
                g[i][j] = next(walls)
            g[i][conn] = v; v += 1                     # ô nối nằm trên đường rắn
    return g

[lo=0, hi=5] mid=2: max hàng = 12 tại cột 3, ô dưới = 13 -> xuống (lo = mid + 1)
[lo=3, hi=5] mid=4: max hàng = 20 tại cột 3, ô dưới = 21 -> xuống (lo = mid + 1)
kết thúc ở hàng 5, cột 0: giá trị 24, đỉnh? True


In [12]:
trace_rows(examples[1][0])

[lo=0, hi=2] mid=1: max hàng = 30 tại cột 1, ô dưới = 16 -> lên/giữ (hi = mid)
[lo=0, hi=1] mid=0: max hàng = 20 tại cột 1, ô dưới = 30 -> xuống (lo = mid + 1)
kết thúc ở hàng 1, cột 1: giá trị 30, đỉnh? True


[1, 1]

In [13]:
def trace_climb(mat, start=(0, 0)):
    m, n = len(mat), len(mat[0]); i, j = start; path = [(i, j, mat[i][j])]
    while True:
        best = (mat[i][j], i, j)
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and mat[a][b] > best[0]:
                best = (mat[a][b], a, b)
        if (best[1], best[2]) == (i, j): break
        i, j = best[1], best[2]; path.append((i, j, mat[i][j]))
    print("leo đồi:", " -> ".join(f"({a},{b})={v}" for a, b, v in path))

SW = snake_walls(5, 4)
for row in SW: print(row)
trace_climb(SW)

[21, 22, 23, 24]
[1, 2, 3, 25]
[29, 28, 27, 26]
[30, 6, 7, 8]
[31, 32, 33, 34]
leo đồi: (0,0)=21 -> (0,1)=22 -> (0,2)=23 -> (0,3)=24 -> (1,3)=25 -> (2,3)=26 -> (2,2)=27 -> (2,1)=28 -> (2,0)=29 -> (3,0)=30 -> (4,0)=31 -> (4,1)=32 -> (4,2)=33 -> (4,3)=34


Với ma trận "rắn có tường" (hàng lẻ là tường giá trị nhỏ, chỉ chừa một ô nối), leo đồi từ `(0, 0)` buộc phải đi hết đường rắn — số bước tỉ lệ với `m n`. Trong khi đó nhị phân trên hàng chỉ cần `⌈log₂ m⌉` lần tìm max hàng. (Với ma trận `M` ở trên, leo đồi đi tắt theo cột nên nhanh — leo đồi chỉ chậm ở những ma trận "xấu" như `SW`.)

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def findPeakGrid(self, mat: List[List[int]]) -> List[int]:
        lo, hi = 0, len(mat) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            row = mat[mid]
            c = row.index(max(row))
            if row[c] < mat[mid + 1][c]:
                lo = mid + 1
            else:
                hi = mid
        row = mat[lo]
        return [lo, row.index(max(row))]

sol = Solution()
for mat, ok in examples:
    got = sol.findPeakGrid(mat)
    print(got, "hợp lệ" if is_peak(mat, *got) else "SAI", "| trong danh sách đáp án mẫu?", got in ok)

[0, 1] hợp lệ | trong danh sách đáp án mẫu? True
[1, 1] hợp lệ | trong danh sách đáp án mẫu? True


## 10. Vì sao thuật toán đúng

**Bất biến.** Trước mỗi vòng, vùng hàng `[lo, hi]` thỏa: (a) nếu `lo > 0` thì tồn tại cột `c₁` với `mat[lo][c₁] > max(mat[lo−1])`; (b) nếu `hi < m−1` thì `max(mat[hi]) > mat[hi+1][c₂]` với `c₂ = argmax(mat[hi])`. Ban đầu `lo = 0`, `hi = m−1` nên đúng hiển nhiên.

**Khi `lo == hi`, ô `(lo, c)` với `c = argmax(mat[lo])` là đỉnh:**
- lớn hơn trái/phải: vì là max hàng;
- lớn hơn ô trên `(lo−1, c)`: (a) cho `max(mat[lo]) ≥ mat[lo][c₁] > max(mat[lo−1]) ≥ mat[lo−1][c]`;
- lớn hơn ô dưới `(lo+1, c)`: (b) với `hi = lo` cho `mat[lo][c] > mat[lo+1][c]` (`c₂ = c` vì cùng được tính bằng `row.index(max(row))`).
⇒ `(lo, c)` là đỉnh.

**Bước giữ bất biến.** Với `mid`, `c = argmax(mat[mid])`:
- Nếu `mat[mid][c] < mat[mid+1][c]` ⇒ `lo = mid + 1`: điều kiện (a) mới đúng với `c₁ = c` vì `mat[mid+1][c] > mat[mid][c] = max(mat[mid])`.
- Ngược lại `mat[mid][c] > mat[mid+1][c]` (không bằng vì ô kề khác nhau) ⇒ `hi = mid`: điều kiện (b) mới đúng với `c₂ = c`.

Mỗi vòng thu hẹp `[lo, hi]` ⇒ kết thúc sau `O(log m)` vòng với `lo == hi`, trả về đỉnh như trên. ∎

## 11. Phân tích Time / Space

Gọi `m` số hàng, `n` số cột (`≤ 500`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quét mọi ô | `O(m n)` | `O(1)` | đơn giản |
| 2. Max toàn cục | `O(m n)` | `O(1)` | luôn đúng |
| 3. Leo đồi | `O(m n)` worst | `O(1)` | nhanh trung bình |
| 4. Nhị phân hàng | `O(n log m)` | `O(1)` | **nộp bài** |
| 5. Nhị phân chiều dài | `O(min · log max)` | `O(1)` | tối ưu theo hình dạng |

In [15]:
def random_grid(m, n, rng):
    # giá trị phân biệt -> chắc chắn không có hai ô kề bằng nhau
    vals = rng.sample(range(1, 10**6), m * n)
    return [vals[i * n:(i + 1) * n] for i in range(m)]

def snake(m, n):
    """Ma trận 'rắn' tăng dần — worst case cho leo đồi từ (0,0)."""
    g = [[0] * n for _ in range(m)]; v = 1
    for i in range(m):
        cols = range(n) if i % 2 == 0 else range(n - 1, -1, -1)
        for j in cols:
            g[i][j] = v; v += 1
    return g

rng = random.Random(1)
G = random_grid(500, 500, rng); S = snake_walls(499, 500)
for name, fn in [("scan", find_peak_scan), ("global", find_peak_global_max), ("rows", find_peak_rows),
                 ("cols", find_peak_cols), ("submit", Solution().findPeakGrid)]:
    t0 = time.perf_counter(); a = fn(G); b = fn(S)
    assert is_peak(G, *a) and is_peak(S, *b)
    print(f"{name:6s}: {time.perf_counter() - t0:.4f}s")
t0 = time.perf_counter(); r, st = find_peak_climb(S)
print(f"climb trên rắn có tường 499x500: {st} bước, {time.perf_counter() - t0:.3f}s")

scan  : 0.0792s
global: 0.0132s
rows  : 0.0002s
cols  : 0.0013s
submit: 0.0001s
climb trên rắn có tường 499x500: 125248 bước, 0.064s


## 12. Bộ test

Bài nhiều đáp án ⇒ kiểm tra bằng `is_peak`. Sinh ma trận đúng ràng buộc "không hai ô kề bằng nhau" bằng hai cách: giá trị phân biệt, hoặc giá trị nhỏ có lặp nhưng sửa để ô kề khác nhau.

In [16]:
def grid_no_adjacent_equal(m, n, rng, V=5):
    g = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            while True:
                v = rng.randint(1, V)
                if (i == 0 or g[i - 1][j] != v) and (j == 0 or g[i][j - 1] != v):
                    g[i][j] = v; break
    return g

fns = [find_peak_scan, find_peak_global_max, lambda g: find_peak_climb(g)[0], find_peak_rows, find_peak_cols,
       find_peak_best_dim, lambda g: Solution().findPeakGrid(g)]

for mat, ok in examples:
    for fn in fns:
        r = fn(mat)
        assert is_peak(mat, *r) and r in ok

edge = [[[5]], [[1, 2]], [[2], [1]], [[1, 2, 1]], [[3], [1], [2]], snake(1, 6), snake(6, 1), snake(4, 4)]
for mat in edge:
    for fn in fns:
        assert is_peak(mat, *fn(mat)), (fn, mat)

rng = random.Random(2024)
for _ in range(2000):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    mat = random_grid(m, n, rng) if rng.random() < 0.5 else grid_no_adjacent_equal(m, n, rng, V=rng.choice([2, 3, 10]))
    for fn in fns:
        r = fn(mat)
        assert is_peak(mat, *r), (fn, mat, r)
print("ví dụ + edge + 2000 ma trận ngẫu nhiên: mọi cách trả về một đỉnh hợp lệ")

ví dụ + edge + 2000 ma trận ngẫu nhiên: mọi cách trả về một đỉnh hợp lệ


In [17]:
# số đỉnh trong ma trận ngẫu nhiên và việc các cách có thể trả về đỉnh KHÁC nhau
rng = random.Random(9)
mat = random_grid(6, 6, rng)
peaks = [[i, j] for i in range(6) for j in range(6) if is_peak(mat, i, j)]
print("các đỉnh:", peaks)
print("rows:", find_peak_rows(mat), "| cols:", find_peak_cols(mat), "| global:", find_peak_global_max(mat), "| scan:", find_peak_scan(mat))

các đỉnh: [[2, 0], [2, 5], [4, 1], [4, 3]]
rows: [2, 5] | cols: [4, 3] | global: [4, 3] | scan: [2, 0]


Các cách có thể trả về **những đỉnh khác nhau** — đều đúng. Đây là lý do test phải dùng `is_peak`, không so sánh với một đáp án cố định.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Nhị phân trên hàng nhưng so sánh **phần tử giữa hàng** thay vì **max hàng**

In [18]:
def wrong_mid_column(mat):
    lo, hi = 0, len(mat) - 1; c = len(mat[0]) // 2
    while lo < hi:
        mid = (lo + hi) // 2
        if mat[mid][c] < mat[mid + 1][c]: lo = mid + 1      # SAI: không dùng max hàng
        else: hi = mid
    return [lo, c]

rng = random.Random(3)
for _ in range(5000):
    mat = random_grid(rng.randint(2, 5), rng.randint(2, 5), rng)
    r = wrong_mid_column(mat)
    if not is_peak(mat, *r):
        print("phản ví dụ:", mat, "-> trả về", r, "không phải đỉnh"); break

phản ví dụ: [[863577, 907572], [282520, 495714], [623641, 753742], [964855, 921503]] -> trả về [3, 1] không phải đỉnh


Chỉ so sánh trên một cột cố định thì ô trả về có thể nhỏ hơn láng giềng trái/phải. **Sửa:** dùng cột của **max hàng**.

### Lỗi 2 — Tìm max hàng `mid` nhưng trả về cột của hàng **khác** khi kết thúc

In [19]:
def wrong_stale_column(mat):
    lo, hi = 0, len(mat) - 1; c = 0
    while lo < hi:
        mid = (lo + hi) // 2; row = mat[mid]; c = row.index(max(row))
        if row[c] < mat[mid + 1][c]: lo = mid + 1
        else: hi = mid
    return [lo, c]                                          # SAI: c là của hàng mid cuối, không phải hàng lo

rng = random.Random(4)
for _ in range(5000):
    mat = random_grid(rng.randint(2, 6), rng.randint(2, 6), rng)
    r = wrong_stale_column(mat)
    if not is_peak(mat, *r):
        print("phản ví dụ:", mat, "-> trả về", r); break

phản ví dụ: [[108178, 756251, 415298, 502141], [162501, 94477, 69747, 20780], [421099, 576090, 962546, 303433]] -> trả về [2, 0]


**Sửa:** sau vòng lặp, tính lại `argmax(mat[lo])`.

### Lỗi 3 — Coi "lớn hơn hoặc bằng" là đỉnh / quên biên `−1`

Hàm kiểm tra viết `nb > v` (thay vì `nb >= v`) chấp nhận ô bằng láng giềng — với ràng buộc "không kề bằng nhau" thì không lộ, nhưng quên xử lý biên (bỏ qua láng giềng ngoài lưới) thì vẫn đúng vì biên là `−1`. Lỗi thật hay gặp: **so sánh với láng giềng ngoài lưới bằng chỉ số âm** của Python.

In [20]:
def wrong_negative_index(mat, i, j):
    v = mat[i][j]
    nbs = [mat[i - 1][j] if i - 1 < len(mat) else -1,       # SAI: i - 1 = -1 hợp lệ trong Python (hàng cuối)
           mat[i + 1][j] if i + 1 < len(mat) else -1,
           mat[i][j - 1] if j - 1 < len(mat[0]) else -1,
           mat[i][j + 1] if j + 1 < len(mat[0]) else -1]
    return all(nb < v for nb in nbs)

mat = [[5, 1], [2, 3], [7, 4]]
print("ô (0,0)=5: kiểm tra sai =", wrong_negative_index(mat, 0, 0), "| đúng =", is_peak(mat, 0, 0),
      "(mat[-1][0] = 7 là hàng cuối, không phải viền -1)")

ô (0,0)=5: kiểm tra sai = False | đúng = True (mat[-1][0] = 7 là hàng cuối, không phải viền -1)


**Sửa:** kiểm tra `0 ≤ a < m` và `0 ≤ b < n` (hàm `get`/`is_peak`).

### Lỗi 4 — Leo đồi đi sang láng giềng lớn hơn **đầu tiên** nhưng có thể vòng lặp nếu so sánh `≥`

In [21]:
def wrong_climb_ge(mat, max_steps=50):
    m, n = len(mat), len(mat[0]); i = j = 0; steps = 0
    while steps < max_steps:
        moved = False
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and mat[a][b] >= mat[i][j]:   # SAI: >= có thể đi qua lại giữa hai ô bằng nhau
                i, j = a, b; moved = True; break
        if not moved: return [i, j], steps
        steps += 1
    return [i, j], steps

print("ma trận có ô kề bằng nhau (vi phạm ràng buộc, chỉ để minh họa):", wrong_climb_ge([[3, 3], [1, 1]]))

ma trận có ô kề bằng nhau (vi phạm ràng buộc, chỉ để minh họa): ([0, 0], 50)


Với `≥`, hai ô kề bằng nhau làm leo đồi đi qua lại mãi (dừng do giới hạn 50 bước). Đề bảo đảm không có ô kề bằng nhau, nhưng dùng `>` vẫn là thói quen đúng. **Sửa:** chỉ đi khi **lớn hơn nghiêm ngặt**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm **một** đỉnh / cực trị địa phương trong `O(log)`" ⇒ nhị phân theo hướng **tăng**: luôn có đỉnh ở phía lớn hơn.
- 2D ⇒ **nén một chiều** bằng max (hàng hoặc cột), nhị phân trên chiều còn lại.
- Bài nhiều đáp án ⇒ viết hàm **kiểm tra hợp lệ** trước, test mọi cách bằng nó.
- Python: cẩn thận chỉ số âm khi xét láng giềng.
- Họ bài: 162 *Find Peak Element*, 852 *Peak Index in a Mountain Array*, 1095 *Find in Mountain Array*.

## 15. Biến thể của bài

1. **162 (1D):** `peak_1d` ở mục 3.2.
2. **Tìm cực tiểu địa phương** (nhỏ hơn mọi láng giềng): đổi chiều so sánh (hoặc đổi dấu ma trận).
3. **Liệt kê mọi đỉnh**: `O(m n)` quét.
4. **3D (khối `m × n × p`)**: nén hai chiều bằng max của "lát", nhị phân trên chiều còn lại — `O(n p log m)`.
5. **Ràng buộc "không kề bằng nhau" bị bỏ**: đỉnh nghiêm ngặt có thể không tồn tại (ma trận hằng) ⇒ phải trả về "không có".

In [22]:
def find_valley_grid(mat):
    """Biến thể 2: cực tiểu địa phương (biên coi như +inf), dùng đổi dấu."""
    neg = [[-v for v in row] for row in mat]
    lo, hi = 0, len(neg) - 1
    while lo < hi:
        mid = (lo + hi) // 2; row = neg[mid]; c = row.index(max(row))
        if row[c] < neg[mid + 1][c]: lo = mid + 1
        else: hi = mid
    return [lo, neg[lo].index(max(neg[lo]))]

def is_valley(mat, i, j):
    m, n = len(mat), len(mat[0])
    return all(not (0 <= i + di < m and 0 <= j + dj < n) or mat[i + di][j + dj] > mat[i][j]
               for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)))

rng = random.Random(5)
for _ in range(1000):
    mat = random_grid(rng.randint(1, 8), rng.randint(1, 8), rng)
    assert is_valley(mat, *find_valley_grid(mat))
print("biến thể cực tiểu địa phương: OK;", find_valley_grid(examples[1][0]))

biến thể cực tiểu địa phương: OK; [0, 0]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét | kiểm 4 láng giềng | `O(mn)` | `O(1)` | kiểm chứng |
| 2. Max toàn cục | max luôn là đỉnh | `O(mn)` | `O(1)` | đơn giản nhất |
| 3. Leo đồi | đi về phía lớn hơn | `O(mn)` worst | `O(1)` | thường nhanh |
| 4. Nhị phân hàng | max hàng + so trên/dưới | `O(n log m)` | `O(1)` | **nộp bài** |
| 5. Chiều dài | nhị phân chiều dài | `O(min·log max)` | `O(1)` | lưới dẹt |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** với `mat = [[1, 2, 3], [6, 5, 4], [7, 8, 9]]`, `findPeakGrid` (Cách 4) trả về gì? Có những đỉnh nào?

**Bài 2 (tính tay):** trace Cách 4 trên `[[10, 1], [2, 20], [30, 3], [4, 40]]`.

**Bài 3 (điền TODO):** hoàn thiện `all_peaks(mat)` trả về danh sách mọi đỉnh.

**Bài 4 (tự viết lại):** viết `find_peak_1d_rows(mat)` — áp **đúng khuôn** `peak_1d` (mục 3.2) lên mảng `rowmax[i] = max(mat[i])` được tính **lười** (chỉ tính khi cần).

**Bài 5 (mini challenge):** biến thể 4 — tìm một đỉnh trong khối 3D `m × n × p` (6 láng giềng, giá trị phân biệt) trong `O(n p log m)`; kiểm chứng bằng hàm `is_peak_3d`. (Biến thể tự đặt.)

In [23]:
def all_peaks(mat):
    # TODO: quét mọi ô, dùng is_peak
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `lo = 0, hi = 2, mid = 1`: max hàng 1 là `6` (cột 0), dưới là `7 > 6` ⇒ `lo = 2`. (2) Kết thúc hàng 2: max `9` ở cột 2. (3) Đỉnh: `9` (và kiểm tra `6`? `7 > 6` nên không). Đáp án `[2, 2]`.

**Bài 2:** (1) `mid = 1`: max hàng 1 là `20` (cột 1), dưới `3 < 20` ⇒ `hi = 1`. (2) `mid = 0`: max `10` (cột 0), dưới `2 < 10` ⇒ `hi = 0`. (3) Trả `[0, 0]` (10 > 1 và 10 > 2 ✓).

**Bài 3:** (1) Hai vòng `for`. (2) `if is_peak(mat, i, j)`. (3) Danh sách `[i, j]`.

**Bài 4:** (1) `rowmax = lambda i: max(mat[i])`. (2) `peak_1d` với `a[mid] < a[mid+1]` thay bằng so sánh **cùng cột** `mat[mid][c] < mat[mid+1][c]` — chú ý: so `rowmax(mid) < rowmax(mid+1)` cũng đúng (vì max hàng dưới `≥` ô cùng cột)! Hãy tự kiểm chứng. (3) Trả cột argmax của hàng cuối.

**Bài 5:** (1) Lát `i` là ma trận `n × p`. (2) `(c1, c2) = argmax` của lát `mid`; so với `cube[mid+1][c1][c2]`. (3) Nhị phân như 2D.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
mat1 = [[1, 2, 3], [6, 5, 4], [7, 8, 9]]
assert Solution().findPeakGrid(mat1) == [2, 2]

# Bài 2
trace_rows([[10, 1], [2, 20], [30, 3], [4, 40]])
assert Solution().findPeakGrid([[10, 1], [2, 20], [30, 3], [4, 40]]) == [0, 0]

# Bài 3
def all_peaks_ans(mat):
    return [[i, j] for i in range(len(mat)) for j in range(len(mat[0])) if is_peak(mat, i, j)]
print("Bài 1 — mọi đỉnh:", all_peaks_ans(mat1))

# Bài 4 — so sánh max hàng với max hàng kế tiếp
def find_peak_1d_rows(mat):
    lo, hi = 0, len(mat) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if max(mat[mid]) < max(mat[mid + 1]):
            lo = mid + 1
        else:
            hi = mid
    return [lo, mat[lo].index(max(mat[lo]))]

rng = random.Random(251)
for _ in range(2000):
    mat = random_grid(rng.randint(1, 8), rng.randint(1, 8), rng)
    assert is_peak(mat, *find_peak_1d_rows(mat))
    assert Solution().findPeakGrid(mat) in all_peaks_ans(mat)
print("Bài 1-4: OK (so max hàng với max hàng kế tiếp cũng đúng)")

[lo=0, hi=3] mid=1: max hàng = 20 tại cột 1, ô dưới = 3 -> lên/giữ (hi = mid)
[lo=0, hi=1] mid=0: max hàng = 10 tại cột 0, ô dưới = 2 -> lên/giữ (hi = mid)
kết thúc ở hàng 0, cột 0: giá trị 10, đỉnh? True
Bài 1 — mọi đỉnh: [[2, 2]]
Bài 1-4: OK (so max hàng với max hàng kế tiếp cũng đúng)


In [25]:
# Bài 5 — đỉnh 3D
def is_peak_3d(cube, i, j, k):
    m, n, p = len(cube), len(cube[0]), len(cube[0][0]); v = cube[i][j][k]
    for di, dj, dk in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
        a, b, c = i + di, j + dj, k + dk
        if 0 <= a < m and 0 <= b < n and 0 <= c < p and cube[a][b][c] >= v:
            return False
    return True

def find_peak_3d(cube):
    def argmax2(sl):
        best = max((v, j, k) for j, row in enumerate(sl) for k, v in enumerate(row))
        return best[1], best[2]
    lo, hi = 0, len(cube) - 1
    while lo < hi:
        mid = (lo + hi) // 2; j, k = argmax2(cube[mid])
        if cube[mid][j][k] < cube[mid + 1][j][k]: lo = mid + 1
        else: hi = mid
    j, k = argmax2(cube[lo])
    return [lo, j, k]

rng = random.Random(252)
for _ in range(500):
    m, n, p = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)
    vals = rng.sample(range(1, 10**6), m * n * p)
    cube = [[vals[(i * n + j) * p:(i * n + j) * p + p] for j in range(n)] for i in range(m)]
    assert is_peak_3d(cube, *find_peak_3d(cube))
print("Bài 5: đỉnh 3D hợp lệ trên 500 khối ngẫu nhiên")

Bài 5: đỉnh 3D hợp lệ trên 500 khối ngẫu nhiên


## 20. Template nhớ nhanh + Checklist

```python
# Đỉnh 2D: nhị phân trên hàng, nén mỗi hàng bằng max
lo, hi = 0, m - 1
while lo < hi:
    mid = (lo + hi) // 2
    c = argmax(mat[mid])
    if mat[mid][c] < mat[mid + 1][c]:   # phía dưới lớn hơn -> có đỉnh ở nửa dưới
        lo = mid + 1
    else:
        hi = mid
return [lo, argmax(mat[lo])]            # tính lại argmax cho hàng cuối cùng
```

**Checklist trước khi submit:**
- [ ] So sánh tại cột của **max hàng**, không phải cột cố định.
- [ ] Sau vòng lặp, tính lại `argmax` của hàng `lo`.
- [ ] `mid + 1` luôn hợp lệ vì `mid < hi`.
- [ ] Test bằng hàm `is_peak` (nhiều đáp án đúng).
- [ ] Không dùng chỉ số âm khi xét láng giềng.